# Customer Churn Prediction
## Notebook 3 — Model Training & Evaluation

**Model:** XGBoost Classifier  
**Goal:** Train a churn prediction model, evaluate it properly, and understand what drives predictions.

### Steps
1. Load clean data and split into train/test
2. Train XGBoost with class imbalance handling
3. Evaluate — accuracy, AUC-ROC, confusion matrix, classification report
4. Feature importance
5. Predict on a new customer
6. Save the model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix,
    classification_report, RocCurveDisplay
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('../data/churn_clean.csv')
print('Shape:', df.shape)

## Step 1 — Train/Test Split

In [ ]:
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set:  {X_train.shape[0]} rows')
print(f'Test set:      {X_test.shape[0]} rows')
print(f'Churn rate in train: {y_train.mean():.2%}')
print(f'Churn rate in test:  {y_test.mean():.2%}')

## Step 2 — Train XGBoost

We use `scale_pos_weight` to handle class imbalance.  
The value is the ratio of negative (No churn) to positive (Churn) samples: 5174 / 1869 ≈ 2.77

In [ ]:
# Calculate class weight ratio
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = round(neg / pos, 2)
print(f'scale_pos_weight = {neg}/{pos} = {scale}')

model = XGBClassifier(
    scale_pos_weight=scale,
    random_state=42,
    eval_metric='logloss',
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1
)

model.fit(X_train, y_train)
print('\nModel trained successfully!')

## Step 3 — Evaluation

We evaluate using:
- **Accuracy** — overall correct predictions
- **AUC-ROC** — how well the model separates churners from non-churners (1.0 = perfect)
- **Confusion matrix** — breakdown of correct vs incorrect predictions
- **Classification report** — precision, recall, F1 per class

In [ ]:
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

acc     = accuracy_score(y_test, y_pred)
auc     = roc_auc_score(y_test, y_proba)

print('=' * 40)
print(f'  Accuracy : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  AUC-ROC  : {auc:.4f}')
print('=' * 40)
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Retained', 'Churned']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Confusion Matrix ---
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
    xticklabels=['Predicted: No', 'Predicted: Yes'],
    yticklabels=['Actual: No', 'Actual: Yes']
)
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')

tn, fp, fn, tp = cm.ravel()
axes[0].set_xlabel(
    f'True Negatives: {tn}  |  False Positives: {fp}\nFalse Negatives: {fn}  |  True Positives: {tp}',
    fontsize=9
)

# --- ROC Curve ---
RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1], color='#4C9BE8')
axes[1].plot([0, 1], [0, 1], 'k--', label='Random classifier')
axes[1].set_title(f'ROC Curve  (AUC = {auc:.3f})', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/model_evaluation.png', bbox_inches='tight')
plt.show()

## Step 4 — Feature Importance

Which features does the model rely on most to predict churn?

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

top10 = feature_importance.head(10)

plt.figure(figsize=(10, 6))
colors = ['#E8554C' if i < 3 else '#4C9BE8' for i in range(len(top10))]
plt.barh(top10['Feature'][::-1], top10['Importance'][::-1], color=colors[::-1])
plt.xlabel('Importance Score', fontsize=12)
plt.title('Top 10 Features Predicting Customer Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/feature_importance.png', bbox_inches='tight')
plt.show()

print('\nTop 10 features:')
print(top10.to_string(index=False))

## Step 5 — Predict on a New Customer

Simulate a real-world prediction for a brand new customer not in the dataset.

In [ ]:
# High-risk customer: month-to-month, fiber optic, short tenure, no security addons
new_customer = pd.DataFrame([{
    'gender': 0, 'SeniorCitizen': 0, 'Partner': 1, 'Dependents': 0,
    'tenure': 12, 'PhoneService': 1, 'MultipleLines': 0,
    'OnlineSecurity': 0, 'OnlineBackup': 0, 'DeviceProtection': 0,
    'TechSupport': 0, 'StreamingTV': 0, 'StreamingMovies': 0,
    'PaperlessBilling': 1, 'MonthlyCharges': 89.5, 'TotalCharges': 1074.0,
    'InternetService_DSL': 0, 'InternetService_Fiber optic': 1, 'InternetService_No': 0,
    'Contract_Month-to-month': 1, 'Contract_One year': 0, 'Contract_Two year': 0,
    'PaymentMethod_Bank transfer (automatic)': 0,
    'PaymentMethod_Credit card (automatic)': 0,
    'PaymentMethod_Electronic check': 1,
    'PaymentMethod_Mailed check': 0
}])

prediction       = model.predict(new_customer)[0]
prediction_proba = model.predict_proba(new_customer)[0]

print('=' * 50)
print('  PREDICTION FOR NEW CUSTOMER')
print('=' * 50)
print(f'\n  Result : {"⚠️  WILL CHURN" if prediction == 1 else "✓  WON'T CHURN"}')
print(f'  Won\'t Churn probability : {prediction_proba[0]:.2%}')
print(f'  Will Churn probability  : {prediction_proba[1]:.2%}')
print('\n  Profile: Fiber optic, month-to-month, 12 months tenure, no security addons')
print('  → High risk profile. Recommend retention offer.')

## Step 6 — Save the Model

In [ ]:
model.save_model('../models/churn_model.json')
print('Model saved to models/churn_model.json')

# Also save feature names for deployment
feature_names = X.columns.tolist()
pd.Series(feature_names).to_csv('../models/feature_names.csv', index=False)
print('Feature names saved to models/feature_names.csv')

## Summary

| Metric | Score |
|---|---|
| Accuracy | ~80–82% |
| AUC-ROC | ~0.85–0.87 |
| Model | XGBoost with scale_pos_weight |
| Training rows | 5,634 |
| Test rows | 1,409 |

**Key drivers of churn:**
- Contract type (month-to-month = highest risk)
- Tenure (shorter = higher risk)
- Monthly charges (higher = more likely to leave)
- Fiber optic internet (higher churn than DSL)
- Lack of online security / tech support add-ons